# 3 - Plusieurs schémas dans un catalogue

Ce notebook illustre la possibilité de regrouper plusieurs jeux de résultats comme **plusieurs schémas d'un même catalogue DuckLake** (un seul `ATTACH`), au lieu d'un catalogue distinct par jeu de résultats.

```
catalogue unique
   ├── schema "predictions"  →  fact_table + metadata + dim_*
   └── schema "shapley"      →  fact_table + metadata + dim_*
```

Chaque classe (`DuckLakeTablesBuilder`, `DatabaseUpdater`, `DatabaseDeleter`, `DatabaseAuditor`, …) accepte un argument `schema` (par défaut `"main"`). Le comportement par défaut reste un schéma unique `main` ; le multi-schéma est activé en passant explicitement `schema=...`.

### Table des matières

0. [Importation des modules](#section_0)
1. [Données synthétiques des deux jeux de résultats](#section_1)
2. [Connexion unique au catalogue](#section_2)
3. [Construction des deux schémas](#section_3)
4. [Mise à jour ciblée d'un schéma](#section_4)
5. [Suppression ciblée d'un schéma](#section_5)
6. [Audit indépendant par schéma](#section_6)
7. [Schéma `main` par défaut](#section_7)
8. [Nettoyage](#section_8)


## 0. Importation des modules <a id="section_0"></a>

In [ ]:
# Rechargement automatique des modules
%load_ext autoreload
%autoreload 2

# Modules de base
import os
import shutil
import sys

import polars as pl

# Ajout du chemin du package
sys.path.append("..")

# Importation des modules ad hoc
from dt_ducklake_manager.connection import DuckLakeConnector
from dt_ducklake_manager.maintenance import DatabaseAuditor, ValidationLevel
from dt_ducklake_manager.operations import DatabaseDeleter, DatabaseUpdater
from dt_ducklake_manager.schema import DuckLakeTablesBuilder

## 1. Données synthétiques des deux jeux de résultats <a id="section_1"></a>

On crée deux jeux de résultats qui partagent une variable catégorielle (`country`) mais diffèrent par leur granularité et leurs colonnes de valeurs :
- `predictions` : les prédictions du modèle ;
- `shapley` : les valeurs de Shapley associées.

In [ ]:
# Seuil catégoriel commun
CATEGORICAL_THRESHOLD = 10

# Jeu de résultats "predictions"
predictions_df = pl.DataFrame(
    {
        "id": [1, 2, 3, 4],
        "country": ["France", "Germany", "France", "Italy"],
        "indicator": ["temperature", "humidity", "pressure", "temperature"],
        "value": [12.4, 0.78, 1013.2, 9.1],
    }
)

# Jeu de résultats "shapley" (granularité et colonnes propres)
shapley_df = pl.DataFrame(
    {
        "id": [1, 2, 3],
        "country": ["France", "Germany", "Spain"],
        "feature": ["lat", "lon", "altitude"],
        "shap_value": [0.21, -0.05, 0.33],
    }
)

predictions_df, shapley_df

## 2. Connexion unique au catalogue <a id="section_2"></a>

Un **seul** catalogue est attaché. Le schéma `main` est activé par défaut, mais les builders ciblent ensuite leurs propres schémas (`predictions`, `shapley`) — `DuckLakeConnector` et `build_schema` créent le schéma s'il n'existe pas encore.

In [ ]:
# Chemins du catalogue et des données (un seul catalogue partagé)
CATALOG_PATH = "multi_schema_catalog.ducklake"
DATA_PATH = "multi_schema_data"

# Nettoyage d'un éventuel état précédent pour repartir d'un catalogue propre
for suffix in ["", ".wal"]:
    p = CATALOG_PATH + suffix
    if os.path.exists(p):
        os.remove(p)
if os.path.exists(DATA_PATH):
    shutil.rmtree(DATA_PATH)

# Connexion unique au catalogue (un seul ATTACH)
conn = DuckLakeConnector(CATALOG_PATH, DATA_PATH).connect()
conn

## 3. Construction des deux schémas <a id="section_3"></a>

On construit chaque jeu de résultats dans son propre schéma, **sur la même connexion**, en passant `schema=...` au builder.

In [ ]:
# Construction du schéma "predictions"
DuckLakeTablesBuilder(
    predictions_df,
    categorical_threshold=CATEGORICAL_THRESHOLD,
    primary_keys=["id"],
    connection=conn,
    schema="predictions",
).build_schema()

# Construction du schéma "shapley"
DuckLakeTablesBuilder(
    shapley_df,
    categorical_threshold=CATEGORICAL_THRESHOLD,
    primary_keys=["id"],
    connection=conn,
    schema="shapley",
).build_schema()

# Inventaire des tables par schéma : les deux jeux de résultats coexistent
conn.execute(
    """
    SELECT table_schema, table_name
    FROM information_schema.tables
    WHERE table_schema IN ('predictions', 'shapley')
    ORDER BY table_schema, table_name
    """
).pl()

In [ ]:
# Lecture des données de chaque schéma via des noms qualifiés
print("predictions.fact_table :")
print(conn.execute("SELECT * FROM predictions.fact_table ORDER BY id").pl())
print("\nshapley.fact_table :")
print(conn.execute("SELECT * FROM shapley.fact_table ORDER BY id").pl())

## 4. Mise à jour ciblée d'un schéma <a id="section_4"></a>

Une mise à jour sur `predictions` n'affecte pas `shapley`.

In [ ]:
# Comptages initiaux
pred_before = conn.execute("SELECT COUNT(*) FROM predictions.fact_table").fetchone()[0]
shap_before = conn.execute("SELECT COUNT(*) FROM shapley.fact_table").fetchone()[0]

# Nouvelle observation pour "predictions" uniquement
new_pred = pl.DataFrame(
    {"id": [5], "country": ["Belgium"], "indicator": ["wind_speed"], "value": [22.0]}
)

DatabaseUpdater(
    connection=conn,
    categorical_threshold=CATEGORICAL_THRESHOLD,
    schema="predictions",
).update_database(new_pred, use_transaction=False)

pred_after = conn.execute("SELECT COUNT(*) FROM predictions.fact_table").fetchone()[0]
shap_after = conn.execute("SELECT COUNT(*) FROM shapley.fact_table").fetchone()[0]
print(f"predictions : {pred_before} -> {pred_after} lignes")
print(f"shapley     : {shap_before} -> {shap_after} lignes (inchangé)")

## 5. Suppression ciblée d'un schéma <a id="section_5"></a>

Une suppression sur `shapley` n'affecte pas `predictions`.

In [ ]:
# Suppression d'une ligne dans "shapley" uniquement
DatabaseDeleter(
    connection=conn,
    categorical_threshold=CATEGORICAL_THRESHOLD,
    auto_cleanup=False,
    schema="shapley",
).delete_rows(filters=[("id", "=", 1)], use_transaction=False)

pred_now = conn.execute("SELECT COUNT(*) FROM predictions.fact_table").fetchone()[0]
shap_now = conn.execute("SELECT COUNT(*) FROM shapley.fact_table").fetchone()[0]
print(f"predictions : {pred_now} lignes (inchangé)")
print(f"shapley     : {shap_now} lignes (après suppression)")

## 6. Audit indépendant par schéma <a id="section_6"></a>

Chaque schéma est audité indépendamment via l'argument `schema`.

In [ ]:
# Audit de chaque schéma : aucun problème critique attendu
for schema_name in ("predictions", "shapley"):
    report = DatabaseAuditor(
        connection=conn,
        categorical_threshold=CATEGORICAL_THRESHOLD,
        schema=schema_name,
    ).validate_database(ValidationLevel.STANDARD)
    print(
        f"{schema_name:12s} | problèmes critiques : "
        f"{report.get_critical_issues_count()} | "
        f"tables validées : {sorted(report.tables_validated)}"
    )

## 7. Schéma `main` par défaut <a id="section_7"></a>

Sans argument `schema`, le comportement est strictement équivalent à l'implémentation historique : un unique schéma `main`.

In [ ]:
# Construction d'un jeu de résultats dans le schéma "main" par défaut
DuckLakeTablesBuilder(
    predictions_df,
    categorical_threshold=CATEGORICAL_THRESHOLD,
    primary_keys=["id"],
    connection=conn,  # aucun schema= : cible "main"
).build_schema()

# Les trois schémas coexistent désormais dans le même catalogue
conn.execute(
    """
    SELECT table_schema, COUNT(*) AS n_tables
    FROM information_schema.tables
    WHERE table_schema IN ('main', 'predictions', 'shapley')
    GROUP BY table_schema
    ORDER BY table_schema
    """
).pl()

## 8. Nettoyage <a id="section_8"></a>

In [ ]:
# Fermeture de la connexion et suppression du catalogue de démonstration
conn.close()
for suffix in ["", ".wal"]:
    p = CATALOG_PATH + suffix
    if os.path.exists(p):
        os.remove(p)
if os.path.exists(DATA_PATH):
    shutil.rmtree(DATA_PATH)
print("Nettoyage terminé.")